# Data aggregation for ML training

In [ ]:
import os
import pandas as pd

Load data

In [ ]:
df_matches = pd.read_csv("data/matches_clean.csv")
df_goals = pd.read_csv("data/goals_clean.csv")
df_players = pd.read_csv("data/players_clean.csv")

all_teams = pd.DataFrame({
    "team": pd.concat([
        df_matches["team1"],
        df_matches["team2"]
    ]).unique()
})

Create goal stats

In [ ]:
team_goals = (
    df_goals
    .groupby("scoring_team")
    .size()
    .reset_index(name="total_goals")
)
team_goals.rename(columns={"scoring_team": "team"}, inplace=True)


team_goals = (
    all_teams
    .merge(team_goals, on="team", how="left")
    .fillna({"total_goals": 0})
)

team_goals["total_goals"] = team_goals["total_goals"].astype(int)

team_goals.head()

In [ ]:
players_with_goals = (
    df_goals
    .groupby("scoring_team")["player_id"]
    .nunique()
    .reset_index(name="num_players_with_goals")
)
players_with_goals.rename(columns={"scoring_team": "team"}, inplace=True)

players_with_goals = (
    all_teams
    .merge(players_with_goals, on="team", how="left")
    .fillna({"num_players_with_goals": 0})
)

players_with_goals["num_players_with_goals"] = players_with_goals["num_players_with_goals"].astype(int)


players_with_goals.head()

In [ ]:
player_goal_counts = (
    df_goals
    .groupby(["scoring_team", "player_id"])
    .size()
    .reset_index(name="goals_by_player")
)
player_goal_counts.rename(columns={"scoring_team": "team"}, inplace=True)

max_goals_player = (
    player_goal_counts
    .groupby("team")["goals_by_player"]
    .max()
    .reset_index(name="max_goals_by_single_player")
)

max_goals_player = (
    all_teams
    .merge(max_goals_player, on="team", how="left")
    .fillna({"max_goals_by_single_player": 0})
)
max_goals_player["max_goals_by_single_player"] = max_goals_player["max_goals_by_single_player"].astype(int)

max_goals_player.head()


Just for fun, let's output the top players, scoring the most goals

In [ ]:
max_goals_player_info = (
    player_goal_counts.loc[
        player_goal_counts.groupby("team")["goals_by_player"].idxmax()
    ][["team", "player_id", "goals_by_player"]]
    .rename(columns={"player_id": "top_scorer", "goals_by_player": "max_goals"})
    .reset_index(drop=True)
)

max_goals_player_info = max_goals_player_info.merge(
    df_players[["player_id", "player_name"]],
    left_on="top_scorer",
    right_on="player_id",
    how="left"
)

max_goals_player_info = max_goals_player_info[["team", "top_scorer", "player_name", "max_goals"]]

max_goals_player_info = max_goals_player_info.sort_values(
    by="max_goals",
    ascending=False
).reset_index(drop=True)

max_goals_player_info.head()

In [ ]:
matches_played = (
    df_matches
    .melt(
        id_vars=["match_id"],
        value_vars=["team1", "team2"],
        value_name="team"
    )
    .groupby("team")
    .nunique()["match_id"]
    .reset_index(name="total_matches")
)

matches_played.head()

In [ ]:
matches_with_goals = (
    df_goals
    .groupby("scoring_team")["match_id"]
    .nunique()
    .reset_index(name="matches_with_goals")
)
matches_with_goals.rename(columns={"scoring_team": "team"}, inplace=True)

matches_with_goals = (
    all_teams
    .merge(matches_with_goals, on="team", how="left")
    .fillna({"matches_with_goals": 0})
)
matches_with_goals["matches_with_goals"] = matches_with_goals["matches_with_goals"].astype(int)

matches_with_goals.head()

Merge stats into team features

In [ ]:
team_features = (
    team_goals
    .merge(players_with_goals, on="team", how="left")
    .merge(max_goals_player, on="team", how="left")
    .merge(matches_played, left_on="team", right_on="team", how="left")
    .merge(matches_with_goals, left_on="team", right_on="team", how="left")
)

team_features.head()

In [ ]:
team_features["matches_without_goals"] = (
    team_features["total_matches"] - team_features["matches_with_goals"]
)

team_features["avg_goals_per_match"] = (
    team_features["total_goals"]
    / team_features["total_matches"]
)

team_features.head()

Add two rows per match (for team1 & team2) for ML training

In [ ]:
team1_rows = df_matches.loc[:, ["match_id", "team1", "team2", "team1_score", "team2_score"]].copy()
team1_rows["win"] = (team1_rows["team1_score"] > team1_rows["team2_score"]).astype(int)

team2_rows = (df_matches[["match_id", "team1", "team2", "team1_score", "team2_score"]]
    .rename(columns={
        "team1": "team2",
        "team2": "team1",
        "team1_score": "team2_score",
        "team2_score": "team1_score",
    }).copy())
team2_rows["win"] = (team2_rows["team1_score"] > team2_rows["team2_score"]).astype(int)

df_matches_labelled = pd.concat([team1_rows, team2_rows], ignore_index=True).sort_values(by="match_id").reset_index(drop=True)
df_matches_labelled.drop(columns=["team1_score", "team2_score"], inplace=True)

df_matches_labelled.head(10)

Join team features to matches

In [ ]:
matches_with_features = df_matches_labelled.merge(
    team_features,
    left_on="team1",
    right_on="team",
    how="left",
    suffixes=("", "_team1")
)

matches_with_features = matches_with_features.merge(
    team_features,
    left_on="team2",
    right_on="team",
    how="left",
    suffixes=("_team1", "_team2")
)


matches_with_features["goals_diff"] = matches_with_features["total_goals_team1"] - matches_with_features["total_goals_team2"]
matches_with_features["matches_diff"] = matches_with_features["total_matches_team1"] - matches_with_features["total_matches_team2"]
matches_with_features["avg_goals_diff"] = matches_with_features["avg_goals_per_match_team1"] - matches_with_features["avg_goals_per_match_team2"]
matches_with_features["num_players_with_goals_diff"] = matches_with_features["num_players_with_goals_team1"] - matches_with_features["num_players_with_goals_team2"]
matches_with_features["max_goals_diff"] = matches_with_features["max_goals_by_single_player_team1"] - matches_with_features["max_goals_by_single_player_team2"]

matches_with_features = matches_with_features.drop(columns=["team_team1", "team_team2"], errors="ignore")

matches_with_features.head(10)

Output file

In [ ]:
os.makedirs("output", exist_ok=True)

team_features.to_csv(
    "model/team_stats.csv",
    index=False
)

matches_with_features.to_csv(
    "model/matches_with_features.csv",
    index=False
)